# 🔍 PNCP — Extração e Análise de Itens: Switch

Este notebook:
1. Lê uma planilha CSV/Excel com a coluna `ID PNCP`
2. Para cada ID, consulta a API do PNCP e busca itens de switch
3. Aplica **dois algoritmos** para classificar se o switch é o item principal:
   - **Algoritmo 1:** Regex (regras textuais)
   - **Algoritmo 2:** Ollama (LLM local)
4. Extrai marca/modelo via regex e Ollama
5. Gera relatório comparativo entre os dois algoritmos
6. Exporta CSV com os resultados

## ⚙️ Célula 1 — Configurações

In [ ]:
# ============================================================
#  CONFIGURAÇÕES — edite aqui antes de rodar
# ============================================================

# Caminho para sua planilha (CSV ou Excel)
ARQUIVO_ENTRADA = "licitacoes.xlsx"   # ou "licitacoes.csv"

# Nome da coluna com o ID PNCP
COLUNA_ID_PNCP = "ID PNCP"

# Arquivos de saída
ARQUIVO_SAIDA_DADOS    = "switch_dados_extraidos.xlsx"
ARQUIVO_SAIDA_RELATORIO = "switch_relatorio_comparativo.xlsx"

# Ollama
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"   # troque pelo modelo que você tem instalado (ex: mistral, phi3)

# API PNCP
PNCP_BASE_URL = "https://pncp.gov.br/api/pncp"

# Pausa entre chamadas à API (segundos) — evita rate limit
PAUSA_SEGUNDOS = 0.5

print("✅ Configurações carregadas")

## 📦 Célula 2 — Instalação e Imports

In [ ]:
import subprocess, sys

def instalar(pacote):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

for pkg in ["requests", "pandas", "openpyxl", "tqdm"]:
    instalar(pkg)

import re
import time
import json
import requests
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display

print("✅ Imports OK")

## 🔧 Célula 3 — Funções: Parse do ID PNCP e API

In [ ]:
def parse_id_pncp(id_pncp: str) -> dict | None:
    """
    Converte '14226731000164-1-000018/2025' em
    {'cnpj': '14226731000164', 'sequencial': 18, 'ano': 2025}
    """
    try:
        id_pncp = str(id_pncp).strip()
        partes = id_pncp.split('-')
        if len(partes) < 3:
            return None
        cnpj = partes[0]
        # partes[1] = código da unidade (ignorado na URL)
        seq_ano = partes[2].split('/')
        sequencial = int(seq_ano[0])   # '000018' → 18
        ano = int(seq_ano[1])
        return {'cnpj': cnpj, 'sequencial': sequencial, 'ano': ano}
    except Exception:
        return None


def api_get(url: str) -> dict | list | None:
    """Faz GET na API do PNCP com tratamento de erros."""
    try:
        r = requests.get(url, headers={"accept": "*/*"}, timeout=15)
        if r.status_code == 200:
            return r.json()
        return None
    except Exception:
        return None


def buscar_contratacao(cnpj, ano, sequencial) -> dict | None:
    url = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{sequencial}"
    return api_get(url)


def buscar_itens(cnpj, ano, sequencial) -> list:
    url = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens"
    resultado = api_get(url)
    if isinstance(resultado, list):
        return resultado
    # Alguns retornos vêm paginados
    if isinstance(resultado, dict) and 'data' in resultado:
        return resultado['data']
    return []


def buscar_resultado_item(cnpj, ano, sequencial, numero_item) -> list:
    url = f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{sequencial}/itens/{numero_item}/resultados"
    resultado = api_get(url)
    if isinstance(resultado, list):
        return resultado
    return []


print("✅ Funções de API carregadas")

## 🔤 Célula 4 — Algoritmo 1: Regex

In [ ]:
# Padrões que indicam que switch é COMPONENTE (não o item principal)
PADROES_COMPONENTE = [
    r'\bcom\s+switch\b',           # "roteador com switch"
    r'\bintegra\w*\s+switch\b',    # "switch integrado"
    r'\binclu\w*\s+switch\b',      # "inclui switch"
    r'\bkit\b.*\bswitch\b',        # "kit ... switch"
    r'\bswitch\b.*\bintegra\w*\b', # "switch integrado"
    r'\bporta\s+switch\b',         # "porta switch"
    r'rack.*switch',               # "rack com switch"
    r'switch.*embutid\w*',         # "switch embutido"
    r'patch\s+panel.*switch',      # "patch panel e switch"
]

# Padrões que confirmam switch como item principal
PADROES_PRINCIPAL = [
    r'^switch\b',                           # começa com switch
    r'^switch\s+\d+\s+portas\b',           # "switch 8 portas"
    r'^switch\s+(gerenci\w+|l[23]|poe\b)',  # switch gerenciável, L2, L3, PoE
    r'\bswitch\b.*\bportas\b',             # switch ... portas
    r'\bswitch\b.*\bmbps\b',               # switch ... mbps
    r'\bswitch\b.*\bgbps\b',               # switch ... gbps
    r'\bswitch\b.*\b10/100\b',             # switch 10/100
    r'\bswitch\b.*\b1000\b',               # switch 1000
]


def regex_eh_switch_principal(descricao: str) -> bool:
    """
    Retorna True se a descrição indica que switch é o item principal.
    Lógica: deve conter 'switch', não ter padrão de componente, e
    idealmente ter padrão de principal.
    """
    texto = descricao.lower().strip()

    if 'switch' not in texto:
        return False

    # Se tiver padrão de componente → não é principal
    for padrao in PADROES_COMPONENTE:
        if re.search(padrao, texto, re.IGNORECASE):
            return False

    # Se tiver padrão de principal → é principal
    for padrao in PADROES_PRINCIPAL:
        if re.search(padrao, texto, re.IGNORECASE):
            return True

    # Caso ambíguo: 'switch' existe mas sem padrão claro
    # Heurística: se switch é uma das primeiras 2 palavras → principal
    palavras = texto.split()
    if palavras and palavras[0] == 'switch':
        return True
    if len(palavras) > 1 and palavras[1] == 'switch':
        return True

    return False


def regex_extrair_marca_modelo(descricao: str) -> dict:
    """Extrai marca e modelo via regex a partir da descrição."""
    resultado = {'marca_regex': None, 'modelo_regex': None}
    texto = descricao.strip()

    # Padrão explícito: MARCA: X / MODELO: Y
    m = re.search(r'marca[:\s]+([A-Z][A-Z0-9\s\-\.]+?)(?:\s+modelo|\s*[;\/]|$)',
                  texto, re.IGNORECASE)
    if m:
        resultado['marca_regex'] = m.group(1).strip()

    m = re.search(r'modelo[:\s]+([A-Z0-9][A-Z0-9\s\-\/\.]+?)(?:\s*[;\/]|\s+[A-Z]{4,}|$)',
                  texto, re.IGNORECASE)
    if m:
        resultado['modelo_regex'] = m.group(1).strip()

    # Marcas conhecidas de switch — fallback quando não tem campo explícito
    MARCAS_CONHECIDAS = [
        'INTELBRAS', 'CISCO', 'HP', 'DELL', 'NETGEAR', 'TP-LINK', 'TPLINK',
        'D-LINK', 'DLINK', 'UBIQUITI', 'MIKROTIK', 'JUNIPER', 'HUAWEI',
        'EXTREME', '3COM', 'ARUBA', 'ZYXEL', 'LINKSYS'
    ]
    if not resultado['marca_regex']:
        for marca in MARCAS_CONHECIDAS:
            if marca.lower() in texto.lower():
                resultado['marca_regex'] = marca
                break

    return resultado


# --- Teste rápido ---
testes = [
    ("SWITCH 08 PORTAS 10/100/1000 MBPS", True),
    ("SWITCH 16 PORTAS 10/100/1000 MBPS", True),
    ("ROTEADOR 1200 MBPS COM SWITCH INTEGRADO 4 PORTAS", False),
    ("KIT RACK COM PATCH PANEL E SWITCH", False),
    ("SWITCH GERENCIÁVEL L2 24 PORTAS INTELBRAS SG 2404 QR", True),
    ("MOUSE USB", False),
]
print("Testes Algoritmo 1 (Regex):")
print(f"{'Descrição':<55} {'Esperado':>8} {'Obtido':>8} {'OK':>4}")
print("-" * 80)
for desc, esperado in testes:
    obtido = regex_eh_switch_principal(desc)
    ok = "✅" if obtido == esperado else "❌"
    print(f"{desc[:54]:<55} {str(esperado):>8} {str(obtido):>8} {ok:>4}")

## 🤖 Célula 5 — Algoritmo 2: Ollama (LLM Local)

In [ ]:
def ollama_classificar_e_extrair(descricao: str) -> dict:
    """
    Usa Ollama para:
    1. Classificar se switch é item principal
    2. Extrair marca e modelo
    Retorna dict com: eh_principal_ollama, marca_ollama, modelo_ollama
    """
    prompt = f"""Você é um especialista em licitações públicas brasileiras.
Analise a descrição de um item de licitação abaixo.

Responda APENAS com um JSON válido, sem texto adicional, sem markdown, sem explicações.
Formato exato:
{{"principal": true/false, "marca": "string ou null", "modelo": "string ou null"}}

Regras:
- "principal": true SOMENTE se o switch for o produto sendo licitado (ex: "SWITCH 8 PORTAS").
  false se switch for componente de outro produto (ex: "ROTEADOR COM SWITCH INTEGRADO") ou parte de kit.
- "marca": nome da fabricante se mencionado, senão null.
- "modelo": código/número do modelo se mencionado, senão null.

Descrição: {descricao}"""

    try:
        r = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
            timeout=30
        )
        if r.status_code != 200:
            return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                    'erro_ollama': f'HTTP {r.status_code}'}

        resposta_texto = r.json().get('response', '').strip()

        # Limpa possíveis blocos markdown que o modelo possa gerar
        resposta_texto = re.sub(r'```json|```', '', resposta_texto).strip()

        # Extrai o JSON da resposta
        match = re.search(r'\{.*\}', resposta_texto, re.DOTALL)
        if not match:
            return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                    'erro_ollama': f'JSON não encontrado: {resposta_texto[:100]}'}

        dados = json.loads(match.group())
        return {
            'eh_principal_ollama': bool(dados.get('principal')),
            'marca_ollama':        dados.get('marca'),
            'modelo_ollama':       dados.get('modelo'),
            'erro_ollama':         None
        }

    except json.JSONDecodeError as e:
        return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                'erro_ollama': f'JSON inválido: {str(e)}'}
    except Exception as e:
        return {'eh_principal_ollama': None, 'marca_ollama': None, 'modelo_ollama': None,
                'erro_ollama': str(e)}


def verificar_ollama() -> bool:
    """Verifica se o Ollama está rodando e o modelo está disponível."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            modelos = [m['name'] for m in r.json().get('models', [])]
            print(f"✅ Ollama online. Modelos disponíveis: {modelos}")
            if not any(OLLAMA_MODEL in m for m in modelos):
                print(f"⚠️  Modelo '{OLLAMA_MODEL}' não encontrado. Troque OLLAMA_MODEL na Célula 1.")
                return False
            return True
    except Exception:
        pass
    print("❌ Ollama offline. Rode: ollama serve")
    return False


OLLAMA_OK = verificar_ollama()

# Teste rápido se Ollama disponível
if OLLAMA_OK:
    print("\nTeste Ollama:")
    res = ollama_classificar_e_extrair("SWITCH 08 PORTAS 10/100/1000 MBPS INTELBRAS SG 800Q")
    print(res)

## 📥 Célula 6 — Carrega a Planilha de Entrada

In [ ]:
# Lê o arquivo (suporta .xlsx, .xls e .csv)
if ARQUIVO_ENTRADA.endswith('.csv'):
    df_entrada = pd.read_csv(ARQUIVO_ENTRADA, dtype=str)
else:
    df_entrada = pd.read_excel(ARQUIVO_ENTRADA, dtype=str)

df_entrada = df_entrada.dropna(subset=[COLUNA_ID_PNCP])
df_entrada = df_entrada[df_entrada[COLUNA_ID_PNCP].str.strip() != '']
df_entrada = df_entrada.reset_index(drop=True)

print(f"✅ Planilha carregada: {len(df_entrada)} linhas")
print(f"Colunas: {list(df_entrada.columns)}")
display(df_entrada[[COLUNA_ID_PNCP]].head(5))

## 🚀 Célula 7 — Pipeline Principal: Consulta API + Classificação

In [ ]:
resultados = []
erros = []

ids_pncp = df_entrada[COLUNA_ID_PNCP].tolist()

for id_pncp in tqdm(ids_pncp, desc="Processando licitações"):

    # 1. Parse do ID
    params = parse_id_pncp(id_pncp)
    if not params:
        erros.append({'id_pncp': id_pncp, 'erro': 'ID PNCP inválido'})
        continue

    cnpj, ano, seq = params['cnpj'], params['ano'], params['sequencial']

    # 2. Dados da contratação
    contratacao = buscar_contratacao(cnpj, ano, seq)
    time.sleep(PAUSA_SEGUNDOS)

    if not contratacao:
        erros.append({'id_pncp': id_pncp, 'erro': 'Contratação não encontrada na API'})
        continue

    # 3. Itens da contratação
    itens = buscar_itens(cnpj, ano, seq)
    time.sleep(PAUSA_SEGUNDOS)

    if not itens:
        erros.append({'id_pncp': id_pncp, 'erro': 'Nenhum item encontrado'})
        continue

    # 4. Filtra itens que contêm 'switch' na descrição
    itens_com_switch = [
        item for item in itens
        if 'switch' in str(item.get('descricao', '')).lower()
    ]

    if not itens_com_switch:
        continue  # Nenhum item com switch nessa licitação

    # 5. Para cada item com switch — classifica e extrai
    for item in itens_com_switch:
        descricao    = str(item.get('descricao', ''))
        numero_item  = item.get('numeroItem')

        # --- Algoritmo 1: Regex ---
        eh_principal_regex = regex_eh_switch_principal(descricao)
        extracao_regex     = regex_extrair_marca_modelo(descricao)

        # --- Algoritmo 2: Ollama ---
        if OLLAMA_OK:
            extracao_ollama = ollama_classificar_e_extrair(descricao)
            time.sleep(0.2)
        else:
            extracao_ollama = {
                'eh_principal_ollama': None,
                'marca_ollama': None,
                'modelo_ollama': None,
                'erro_ollama': 'Ollama offline'
            }

        # --- Resultado do item (vencedor e preço homologado) ---
        resultados_item = buscar_resultado_item(cnpj, ano, seq, numero_item)
        time.sleep(PAUSA_SEGUNDOS)

        # Pega o primeiro resultado homologado (ordemClassificacao = 1)
        vencedor = next(
            (r for r in resultados_item if r.get('ordemClassificacaoSrp', 999) == 1),
            resultados_item[0] if resultados_item else {}
        )

        # --- Monta linha do resultado ---
        linha = {
            # Identificação
            'id_pncp':               id_pncp,
            'cnpj_orgao':            cnpj,
            'ano':                   ano,
            'sequencial':            seq,
            'numero_item':           numero_item,

            # Dados da contratação
            'orgao':                 contratacao.get('orgaoEntidade', {}).get('razaoSocial', ''),
            'municipio':             contratacao.get('unidadeOrgao', {}).get('municipioNome', ''),
            'uf':                    contratacao.get('unidadeOrgao', {}).get('ufSigla', ''),
            'modalidade':            contratacao.get('modalidadeNome', ''),
            'situacao_contratacao':  contratacao.get('situacaoCompraNome', ''),
            'data_abertura':         contratacao.get('dataAberturaProposta', ''),
            'data_publicacao':       contratacao.get('dataPublicacaoPncp', ''),
            'srp':                   contratacao.get('srp', ''),

            # Dados do item
            'descricao':             descricao,
            'material_servico':      item.get('materialOuServicoNome', ''),
            'quantidade':            item.get('quantidade', ''),
            'unidade_medida':        item.get('unidadeMedida', ''),
            'valor_unitario_est':    item.get('valorUnitarioEstimado', ''),
            'valor_total_est':       item.get('valorTotal', ''),
            'situacao_item':         item.get('situacaoCompraItemNome', ''),
            'tem_resultado':         item.get('temResultado', ''),
            'ncm':                   item.get('ncmNbsCodigo', ''),

            # Resultado (vencedor)
            'cnpj_vencedor':         vencedor.get('niFornecedor', ''),
            'nome_vencedor':         vencedor.get('nomeRazaoSocialFornecedor', ''),
            'porte_vencedor':        vencedor.get('porteFornecedorNome', ''),
            'valor_unit_homologado': vencedor.get('valorUnitarioHomologado', ''),
            'valor_total_homolog':   vencedor.get('valorTotalHomologado', ''),
            'qtd_homologada':        vencedor.get('quantidadeHomologada', ''),
            'data_resultado':        vencedor.get('dataResultado', ''),

            # Algoritmo 1 — Regex
            'switch_principal_regex': eh_principal_regex,
            'marca_regex':            extracao_regex.get('marca_regex'),
            'modelo_regex':           extracao_regex.get('modelo_regex'),

            # Algoritmo 2 — Ollama
            'switch_principal_ollama': extracao_ollama.get('eh_principal_ollama'),
            'marca_ollama':            extracao_ollama.get('marca_ollama'),
            'modelo_ollama':           extracao_ollama.get('modelo_ollama'),
            'erro_ollama':             extracao_ollama.get('erro_ollama'),
        }
        resultados.append(linha)

print(f"\n✅ Processamento concluído!")
print(f"   Itens com 'switch' encontrados: {len(resultados)}")
print(f"   Erros: {len(erros)}")

## 📊 Célula 8 — Análise e Comparação dos Algoritmos

In [ ]:
df = pd.DataFrame(resultados)

if df.empty:
    print("⚠️  Nenhum item com 'switch' encontrado.")
else:
    print(f"Total de itens com 'switch': {len(df)}")
    print()

    # ── Resumo por algoritmo ──────────────────────────────────────
    n_regex   = df['switch_principal_regex'].sum()
    n_ollama  = df['switch_principal_ollama'].sum() if OLLAMA_OK else 'N/A'

    print("=" * 50)
    print("CLASSIFICAÇÃO: switch como item PRINCIPAL")
    print("=" * 50)
    print(f"  Algoritmo 1 (Regex):  {n_regex} de {len(df)} itens ({100*n_regex/len(df):.1f}%)")
    if OLLAMA_OK:
        print(f"  Algoritmo 2 (Ollama): {n_ollama} de {len(df)} itens ({100*n_ollama/len(df):.1f}%)")
    print()

    # ── Concordância entre algoritmos ────────────────────────────
    if OLLAMA_OK:
        df_comp = df.dropna(subset=['switch_principal_ollama'])
        if len(df_comp) > 0:
            concordancia = (df_comp['switch_principal_regex'] == df_comp['switch_principal_ollama']).mean()
            print("=" * 50)
            print("CONCORDÂNCIA ENTRE ALGORITMOS")
            print("=" * 50)
            print(f"  Taxa de concordância: {concordancia*100:.1f}%")

            # Casos onde divergem
            divergencias = df_comp[
                df_comp['switch_principal_regex'] != df_comp['switch_principal_ollama']
            ][['descricao', 'switch_principal_regex', 'switch_principal_ollama']]

            print(f"  Divergências: {len(divergencias)} itens")
            if len(divergencias) > 0:
                print()
                print("Itens com divergência entre algoritmos:")
                display(divergencias.reset_index(drop=True))

    # ── Itens classificados como principal (por regex) ────────────
    print()
    print("=" * 50)
    print("ITENS CLASSIFICADOS COMO SWITCH PRINCIPAL (Regex)")
    print("=" * 50)
    df_principal = df[df['switch_principal_regex'] == True].copy()
    display(df_principal[['descricao', 'quantidade', 'valor_unitario_est',
                           'valor_unit_homologado', 'nome_vencedor',
                           'marca_regex', 'modelo_regex']].reset_index(drop=True))

    # ── Análise de preços (somente itens classificados como principal) ──
    if len(df_principal) > 0:
        df_precos = df_principal.copy()
        for col in ['valor_unitario_est', 'valor_unit_homologado']:
            df_precos[col] = pd.to_numeric(df_precos[col], errors='coerce')

        df_precos = df_precos.dropna(subset=['valor_unit_homologado'])

        if len(df_precos) > 0:
            print()
            print("=" * 50)
            print("ANÁLISE DE PREÇOS — SWITCH PRINCIPAL")
            print("=" * 50)
            stats = df_precos['valor_unit_homologado'].describe()
            print(f"  Menor preço:   R$ {stats['min']:,.2f}")
            print(f"  Maior preço:   R$ {stats['max']:,.2f}")
            print(f"  Média:         R$ {stats['mean']:,.2f}")
            print(f"  Mediana:       R$ {stats['50%']:,.2f}")
            print(f"  Qtd amostras:  {int(stats['count'])}")

            # Desconto médio estimado vs homologado
            df_desc = df_precos.dropna(subset=['valor_unitario_est'])
            df_desc = df_desc[df_desc['valor_unitario_est'] > 0]
            if len(df_desc) > 0:
                df_desc = df_desc.copy()
                df_desc['desconto_pct'] = (
                    (df_desc['valor_unitario_est'] - df_desc['valor_unit_homologado'])
                    / df_desc['valor_unitario_est'] * 100
                )
                print(f"  Desconto médio (est→homolog): {df_desc['desconto_pct'].mean():.1f}%")

## 💾 Célula 9 — Exportação dos Resultados

In [ ]:
if not df.empty:

    # ── Arquivo 1: Todos os dados extraídos ──────────────────────
    df.to_excel(ARQUIVO_SAIDA_DADOS, index=False)
    print(f"✅ Dados exportados: {ARQUIVO_SAIDA_DADOS}")

    # ── Arquivo 2: Relatório comparativo ─────────────────────────
    with pd.ExcelWriter(ARQUIVO_SAIDA_RELATORIO, engine='openpyxl') as writer:

        # Aba 1: Todos os itens com switch
        colunas_resumo = [
            'id_pncp', 'orgao', 'municipio', 'uf', 'descricao',
            'quantidade', 'unidade_medida',
            'valor_unitario_est', 'valor_unit_homologado', 'valor_total_homolog',
            'nome_vencedor', 'cnpj_vencedor', 'porte_vencedor',
            'data_resultado', 'situacao_item', 'srp',
            'switch_principal_regex', 'marca_regex', 'modelo_regex',
            'switch_principal_ollama', 'marca_ollama', 'modelo_ollama',
        ]
        colunas_existentes = [c for c in colunas_resumo if c in df.columns]
        df[colunas_existentes].to_excel(writer, sheet_name='Todos os Itens Switch', index=False)

        # Aba 2: Somente switch principal (regex)
        df_principal = df[df['switch_principal_regex'] == True]
        df_principal[colunas_existentes].to_excel(
            writer, sheet_name='Switch Principal (Regex)', index=False)

        # Aba 3: Somente switch principal (Ollama) — se disponível
        if OLLAMA_OK and 'switch_principal_ollama' in df.columns:
            df_principal_ollama = df[df['switch_principal_ollama'] == True]
            df_principal_ollama[colunas_existentes].to_excel(
                writer, sheet_name='Switch Principal (Ollama)', index=False)

        # Aba 4: Divergências
        if OLLAMA_OK and 'switch_principal_ollama' in df.columns:
            df_div = df[
                df['switch_principal_regex'] != df['switch_principal_ollama']
            ]
            if len(df_div) > 0:
                df_div[['id_pncp', 'descricao',
                         'switch_principal_regex', 'switch_principal_ollama',
                         'erro_ollama']].to_excel(
                    writer, sheet_name='Divergências', index=False)

        # Aba 5: Comparativo de preços
        df_precos = df[df['switch_principal_regex'] == True].copy()
        for col in ['valor_unitario_est', 'valor_unit_homologado']:
            df_precos[col] = pd.to_numeric(df_precos[col], errors='coerce')

        df_precos_clean = df_precos.dropna(subset=['valor_unit_homologado']).copy()
        df_precos_clean['desconto_%'] = (
            (df_precos_clean['valor_unitario_est'] - df_precos_clean['valor_unit_homologado'])
            / df_precos_clean['valor_unitario_est'].replace(0, float('nan')) * 100
        ).round(2)

        cols_precos = ['id_pncp', 'orgao', 'municipio', 'uf', 'descricao',
                       'marca_regex', 'modelo_regex', 'quantidade',
                       'valor_unitario_est', 'valor_unit_homologado', 'desconto_%',
                       'nome_vencedor', 'cnpj_vencedor', 'data_resultado']
        cols_ok = [c for c in cols_precos if c in df_precos_clean.columns]
        df_precos_clean[cols_ok].sort_values('valor_unit_homologado').to_excel(
            writer, sheet_name='Comparativo de Preços', index=False)

        # Aba 6: Erros
        if erros:
            pd.DataFrame(erros).to_excel(writer, sheet_name='Erros', index=False)

    print(f"✅ Relatório exportado: {ARQUIVO_SAIDA_RELATORIO}")
    print()
    print("Abas do relatório:")
    print("  📋 Todos os Itens Switch     — todos os itens onde 'switch' aparece na descrição")
    print("  ✅ Switch Principal (Regex)  — filtrado pelo Algoritmo 1")
    if OLLAMA_OK:
        print("  🤖 Switch Principal (Ollama) — filtrado pelo Algoritmo 2")
        print("  ⚠️  Divergências              — casos onde os algoritmos discordam")
    print("  💰 Comparativo de Preços     — preços ordenados do menor para o maior")
    if erros:
        print("  ❌ Erros                     — IDs que falharam")

else:
    print("⚠️  Nenhum resultado para exportar.")